In [6]:
%%writefile dist_lab.py

import os
import torch
import torch.distributed as dist

dist.init_process_group("gloo")
rank = dist.get_rank()

if rank == 0:
    x = torch.tensor([2.0])
    dist.send(x, dst=1)

    result = torch.empty(1)
    dist.recv(result, src=1)
    print(f"Rank 0: received {result.item()}", flush=True)

else:
    y = torch.tensor([4.0])
    x = torch.empty(1)

    dist.recv(x, src=0)
    result = x + y
    dist.send(result, dst=0)
    print(f"Rank 1: computed {result.item()}", flush=True)

dist.destroy_process_group()

Writing dist_lab.py


In [7]:
!{sys.executable} -m torch.distributed.run --standalone --nproc-per-node=2 dist_lab.py

W1111 20:24:39.295000 13372 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.

*****************************************
Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
*****************************************
Rank 1: computed 6.0
Rank 0: received 6.0
